# Handling Message Blocks

Once tools are in play, a response isn't just text anymore — Claude returns a **multi-block** assistant message. Pass `tools=[...]` and, when Claude decides to use one, `content` becomes a list that can hold:

- a **text** block — human-readable narration ("Let me look that up…")
- a **tool_use** block — `id` (to match the result later), `name` (which function), `input` (args dict), `type: "tool_use"`

**The full 5-step flow:**
```
1. Send user message + tool schemas          ← this lesson
2. Receive assistant msg (text + tool_use)   ← this lesson
3. Extract the tool call, run the function    (next: Sending tool results)
4. Send the tool_result back + full history   (next)
5. Receive Claude's final answer              (next)
```

This lesson does 1–2: call with the schema, inspect the blocks, and preserve them in history. Runs on Haiku 4.5 (tool use works on every current model).

## Setup — client, multi-block-aware helpers, tool + schema

The helpers are **updated** to accept either a plain string *or* a full response's block list. Passing a `Message` object extracts its `.content` (the list of blocks); passing a string wraps it as before. This is what lets us store a multi-block assistant turn in history.

In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from datetime import datetime
from anthropic import Anthropic
from anthropic.types import ToolParam

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    # Accept a raw string OR a Message/blocks; store the block list when given one
    content = message.content if hasattr(message, "content") else message
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, message):
    content = message.content if hasattr(message, "content") else message
    messages.append({"role": "assistant", "content": content})


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)

print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Step 1 — send the question with the tool schema

`tools` takes a list of schemas. We call `client.messages.create` directly (not our `chat` helper) because we need the *whole* response object — `stop_reason` and the block list — not just extracted text.

In [3]:
messages = []
add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print("stop_reason:", response.stop_reason)
response.content

stop_reason: tool_use


[ToolUseBlock(id='toolu_011pR5ejc8uAuxpSYKm7yFTz', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]

## Step 2 — inspect the blocks

`stop_reason` should be `"tool_use"`. Walk the content list and see each block's type; pull the tool_use block's `id`, `name`, and `input`.

In [4]:
for block in response.content:
    print("---", block.type)
    if block.type == "text":
        print(block.text)
    elif block.type == "tool_use":
        print("id:   ", block.id)
        print("name: ", block.name)
        print("input:", block.input)

--- tool_use
id:    toolu_011pR5ejc8uAuxpSYKm7yFTz
name:  get_current_datetime
input: {'date_format': '%H:%M:%S'}


## Preserve the full multi-block message in history

Claude is stateless — *you* keep the history. Critically, append the **entire** `content` (text block **and** tool_use block), or the follow-up request loses context. Our updated helper stores `response.content` when handed the response object.

In [5]:
add_assistant_message(messages, response)

# The assistant turn now carries the tool_use block, ready for the result to be attached next
messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_011pR5ejc8uAuxpSYKm7yFTz', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

## Next

We have the `tool_use` block — its `id`, the `name` to call, and the `input` args. Next lesson (**Sending tool results**): run `get_current_datetime(**input)`, wrap the return value in a `tool_result` block tagged with that same `id`, send it back with the full history, and get Claude's final answer.